# Despliegue Aplicado, Paso a Paso — Del modelo entrenado a una API real
### Tecnologías Emergentes (ISO46B) — Semana 6 (notebook complementario de experimentación)

Notebook **autocontenido** para **Google Colab** (`Entorno de ejecución > Ejecutar todas`). Retoma el modelo de clasificación de lluvia de la Semana 6 y se enfoca por completo en la parte de **despliegue**: levantar un servidor REAL (no simulado), consumirlo con solicitudes HTTP reales, exponerlo temporalmente a internet, y comparar la misma API construida con **Flask** y con **FastAPI**.

**Diferencia con el notebook de la guía de laboratorio:** ahí se probó la API con el *cliente de pruebas* de Flask (`app.test_client()`), que simula solicitudes sin red real — es la forma correcta de hacer pruebas automatizadas. **Aquí** levantamos un servidor de verdad, escuchando en un puerto real, y lo consumimos con la librería `requests` tal como lo haría una aplicación externa — esa es la experiencia "aplicada" de despliegue.


## 0. Configuración inicial

In [ ]:
!pip install -q flask requests fastapi "uvicorn[standard]" nest-asyncio pyngrok

In [ ]:
import time
import json
import threading
import requests
import numpy as np
import pandas as pd

print("Librerías listas.")

## 1. Preparar rápidamente un modelo para desplegar

Usamos el mismo modelo de clasificación de lluvia de la Semana 6 (predecir si lloverá mañana en Huancayo, usando temp_max, temp_min y viento de hoy). Si ya tienes `modelo_produccion.keras` y `escalador_produccion.pkl` de la guía de laboratorio, puedes subirlos a Colab y saltar directamente a la Sección 2. Aquí los regeneramos en pocas celdas para que el notebook sea independiente.

In [ ]:
import datetime as dt

LATITUD, LONGITUD = -12.0654, -75.2049  # Huancayo
fin = dt.date.today() - dt.timedelta(days=5)
inicio = fin - dt.timedelta(days=400)

url = "https://archive-api.open-meteo.com/v1/archive"
parametros = {
    "latitude": LATITUD, "longitude": LONGITUD,
    "start_date": inicio.isoformat(), "end_date": fin.isoformat(),
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
    "timezone": "America/Lima",
}
respuesta = requests.get(url, params=parametros, timeout=30)
respuesta.raise_for_status()
datos = respuesta.json()["daily"]

df = pd.DataFrame(datos).rename(columns={
    "temperature_2m_max": "temp_max", "temperature_2m_min": "temp_min",
    "precipitation_sum": "precipitacion", "windspeed_10m_max": "viento_max",
}).dropna().reset_index(drop=True)

df["lluvia_mañana"] = (df["precipitacion"].shift(-1) > 0).astype(int)
df = df.dropna().reset_index(drop=True).drop(columns=["precipitacion"])

COLUMNAS_CARACTERISTICAS = ["temp_max", "temp_min", "viento_max"]
print(f"Dataset listo: {len(df)} filas")

In [ ]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

X = df[COLUMNAS_CARACTERISTICAS].values
y = df["lluvia_mañana"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_test_esc = escalador.transform(X_test)

modelo = keras.Sequential([
    layers.Input(shape=(3,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(X_train_esc, y_train, epochs=60, batch_size=16, verbose=0)

modelo.save("modelo_produccion.keras")
with open("escalador_produccion.pkl", "wb") as f:
    pickle.dump(escalador, f)

perdida, precision = modelo.evaluate(X_test_esc, y_test, verbose=0)
print(f"Modelo listo para desplegar. Precisión en test: {precision:.3f}")

## 2. Construir la API con Flask (recordatorio)

In [ ]:
%%writefile api_flask.py
import pickle
import numpy as np
from flask import Flask, request, jsonify
from tensorflow import keras

COLUMNAS_CARACTERISTICAS = ["temp_max", "temp_min", "viento_max"]

app = Flask(__name__)
modelo = keras.models.load_model("modelo_produccion.keras")
with open("escalador_produccion.pkl", "rb") as f:
    escalador = pickle.load(f)


def extraer_caracteristicas(datos_json):
    faltantes = [c for c in COLUMNAS_CARACTERISTICAS if c not in datos_json]
    if faltantes:
        raise ValueError(f"Faltan las siguientes características: {faltantes}")
    return np.array([[float(datos_json[c]) for c in COLUMNAS_CARACTERISTICAS]])


@app.route("/predecir", methods=["POST"])
def predecir():
    datos_json = request.get_json(silent=True)
    if datos_json is None:
        return jsonify({"error": "Se esperaba un cuerpo JSON válido"}), 400
    try:
        caracteristicas = extraer_caracteristicas(datos_json)
    except ValueError as error:
        return jsonify({"error": str(error)}), 400

    caracteristicas_esc = escalador.transform(caracteristicas)
    probabilidad = float(modelo.predict(caracteristicas_esc, verbose=0)[0][0])
    prediccion = "lluvia" if probabilidad > 0.5 else "no_lluvia"
    return jsonify({"prediccion": prediccion, "probabilidad": round(probabilidad, 4)})


@app.route("/salud", methods=["GET"])
def salud():
    return jsonify({"estado": "ok"})


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

## 3. Levantar el servidor REAL dentro de Colab

En una terminal normal, `python api_flask.py` levanta el servidor y "bloquea" la consola. Dentro de un notebook no podemos bloquear una celda indefinidamente (nunca podríamos ejecutar la siguiente), así que lanzamos el servidor en un **hilo (thread) en segundo plano** — una técnica real, usada también en despliegues donde un proceso debe convivir con otras tareas.

In [ ]:
import importlib
import api_flask
importlib.reload(api_flask)  # por si ya se había importado antes en esta sesión

def lanzar_servidor():
    api_flask.app.run(host="0.0.0.0", port=5000, use_reloader=False)

hilo_servidor = threading.Thread(target=lanzar_servidor, daemon=True)
hilo_servidor.start()

time.sleep(2)  # darle un momento al servidor para iniciar
print("Servidor Flask corriendo en segundo plano en http://127.0.0.1:5000")

## 4. Consumir la API con solicitudes HTTP REALES

A diferencia del `test_client()` de la guía, aquí `requests.post(...)` viaja por la red local de verdad, tal como lo haría cualquier aplicación externa.

In [ ]:
respuesta = requests.post(
    "http://127.0.0.1:5000/predecir",
    json={"temp_max": 18.5, "temp_min": 4.2, "viento_max": 12.0},
)
print("Código de estado HTTP:", respuesta.status_code)
print("Respuesta:", respuesta.json())

In [ ]:
respuesta_salud = requests.get("http://127.0.0.1:5000/salud")
print(respuesta_salud.status_code, respuesta_salud.json())

respuesta_error = requests.post("http://127.0.0.1:5000/predecir", json={"temp_max": 18.5})
print(respuesta_error.status_code, respuesta_error.json())

## 5. Simular varios "clientes" consultando la API y medir latencia

En un despliegue real, muchas solicitudes distintas (de distintos usuarios o aplicaciones) llegan a la misma API. Simulamos esto con varias solicitudes seguidas y medimos cuánto tarda en responder cada una.

In [ ]:
rng = np.random.default_rng(1)
tiempos_respuesta = []

for _ in range(15):
    entrada = {
        "temp_max": float(rng.normal(20, 4)),
        "temp_min": float(rng.normal(6, 3)),
        "viento_max": float(rng.normal(15, 5)),
    }
    inicio_t = time.time()
    r = requests.post("http://127.0.0.1:5000/predecir", json=entrada)
    tiempos_respuesta.append((time.time() - inicio_t) * 1000)  # ms

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(tiempos_respuesta, marker="o", color="#1F3864")
plt.axhline(np.mean(tiempos_respuesta), color="crimson", linestyle="--", label=f"promedio: {np.mean(tiempos_respuesta):.1f} ms")
plt.title("Latencia de la API por solicitud (servidor de desarrollo, un solo hilo)")
plt.xlabel("N.º de solicitud"); plt.ylabel("Tiempo de respuesta (ms)")
plt.legend()
plt.show()

**Lectura:** el servidor de desarrollo de Flask (el que usamos aquí) atiende las solicitudes de una en una. Esto es aceptable para pruebas y para esta práctica, pero NO para producción con muchos usuarios simultáneos — de ahí la mención, en la guía de laboratorio, de servidores como Gunicorn o Uvicorn, diseñados para atender muchas solicitudes a la vez.

## 6. (Opcional) Exponer la API a internet con ngrok

Hasta ahora la API solo es accesible **dentro** de esta sesión de Colab (`127.0.0.1` = "esta misma máquina"). Para probarla desde tu propio navegador, celular o Postman —como lo harías con una API real en producción— necesitas una URL pública. **ngrok** crea un túnel temporal desde una URL pública hacia este servidor.

**Requiere una cuenta gratuita en ngrok.com** y un *authtoken* personal (Dashboard → "Your Authtoken"). Si no quieres crear una cuenta, puedes omitir esta sección — todo lo importante del despliegue ya se demostró en las Secciones 3 a 5.

In [ ]:
from pyngrok import ngrok

# TU AUTHTOKEN: pégalo aquí entre comillas (Dashboard de ngrok.com -> "Your Authtoken")
NGROK_AUTHTOKEN = "PEGA_AQUI_TU_AUTHTOKEN"

if NGROK_AUTHTOKEN != "PEGA_AQUI_TU_AUTHTOKEN":
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    tunel = ngrok.connect(5000)
    print(f"Tu API es accesible públicamente en: {tunel.public_url}")
    print(f"Pruébala desde tu navegador o Postman: {tunel.public_url}/salud")
else:
    print("Sección omitida: agrega tu authtoken de ngrok.com arriba para activarla.")

In [ ]:
# Si activaste el túnel, puedes probarlo también desde AQUÍ MISMO usando la URL pública:
if NGROK_AUTHTOKEN != "PEGA_AQUI_TU_AUTHTOKEN":
    respuesta_publica = requests.post(
        f"{tunel.public_url}/predecir",
        json={"temp_max": 18.5, "temp_min": 4.2, "viento_max": 12.0},
    )
    print(respuesta_publica.status_code, respuesta_publica.json())

**Importante sobre seguridad:** una URL pública de ngrok, mientras el túnel está activo, puede ser consultada por cualquiera que la conozca. Para esta práctica no es un problema (el modelo no expone datos sensibles), pero en un despliegue real nunca se expone una API sin autenticación (claves de API, tokens, etc.) — es un tema que se profundiza en cursos de ingeniería de software más avanzados.

## 7. La misma API, construida con FastAPI

La guía de laboratorio usó Flask. Aquí construimos el **mismo endpoint** con **FastAPI**, para comparar directamente el código y confirmar lo visto en el insumo de la semana: ambos resuelven el mismo problema, con decisiones de ingeniería distintas (FastAPI valida automáticamente el tipo de los datos de entrada, y genera documentación interactiva sola).

In [ ]:
%%writefile api_fastapi.py
import pickle
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
from tensorflow import keras

COLUMNAS_CARACTERISTICAS = ["temp_max", "temp_min", "viento_max"]

app = FastAPI(title="API de pronóstico de lluvia — Huancayo")
modelo = keras.models.load_model("modelo_produccion.keras")
with open("escalador_produccion.pkl", "rb") as f:
    escalador = pickle.load(f)


class DatosClima(BaseModel):
    temp_max: float
    temp_min: float
    viento_max: float


@app.post("/predecir")
def predecir(datos: DatosClima):
    caracteristicas = np.array([[datos.temp_max, datos.temp_min, datos.viento_max]])
    caracteristicas_esc = escalador.transform(caracteristicas)
    probabilidad = float(modelo.predict(caracteristicas_esc, verbose=0)[0][0])
    prediccion = "lluvia" if probabilidad > 0.5 else "no_lluvia"
    return {"prediccion": prediccion, "probabilidad": round(probabilidad, 4)}


@app.get("/salud")
def salud():
    return {"estado": "ok"}

**Nota sobre la validación automática:** en FastAPI, `DatosClima` declara los tipos esperados (`float`). Si falta un campo o se envía un tipo incorrecto, FastAPI responde automáticamente con un error 422 detallado — sin que el desarrollador tenga que escribir la validación a mano, como sí hicimos en `extraer_caracteristicas` para Flask.

In [ ]:
import nest_asyncio
import uvicorn
import api_fastapi
importlib.reload(api_fastapi)

nest_asyncio.apply()  # permite correr un servidor asíncrono dentro del notebook

def lanzar_servidor_fastapi():
    uvicorn.run(api_fastapi.app, host="0.0.0.0", port=8000, log_level="warning")

hilo_fastapi = threading.Thread(target=lanzar_servidor_fastapi, daemon=True)
hilo_fastapi.start()
time.sleep(2)
print("Servidor FastAPI corriendo en segundo plano en http://127.0.0.1:8000")

In [ ]:
respuesta_fastapi = requests.post(
    "http://127.0.0.1:8000/predecir",
    json={"temp_max": 18.5, "temp_min": 4.2, "viento_max": 12.0},
)
print("Flask  :", requests.post("http://127.0.0.1:5000/predecir", json={"temp_max": 18.5, "temp_min": 4.2, "viento_max": 12.0}).json())
print("FastAPI:", respuesta_fastapi.json())

# El error de validación automática de FastAPI (sin código extra):
respuesta_fastapi_error = requests.post("http://127.0.0.1:8000/predecir", json={"temp_max": 18.5})
print("\nFastAPI ante un campo faltante -> código", respuesta_fastapi_error.status_code)
print(respuesta_fastapi_error.json())

**Documentación automática:** si expusieras el servidor FastAPI con ngrok (Sección 6, usando el puerto 8000 en vez de 5000), podrías visitar `<tu-url>/docs` y ver una interfaz interactiva generada automáticamente — pruébalo si tienes tiempo.

## 8. Apagar los servidores (buena práctica)

Los servidores quedan corriendo en hilos de segundo plano mientras el notebook esté activo. Es buena práctica cerrarlos explícitamente al terminar, igual que cerrarías una conexión a una base de datos.

In [ ]:
from pyngrok import ngrok as ngrok_cierre
ngrok_cierre.kill()  # cierra cualquier túnel de ngrok activo
print("Túneles de ngrok cerrados.")
print("Nota: los hilos de Flask/FastAPI daemon se cierran automáticamente al terminar o reiniciar este notebook.")

## 9. Resumen

| Paso | Qué demuestra |
|---|---|
| Servidor en un hilo en segundo plano | Cómo ejecutar un servicio "de verdad" sin bloquear el resto del trabajo |
| `requests.post(...)` a `127.0.0.1` | Una solicitud HTTP real, no una simulación — así se consume una API en la práctica |
| Medición de latencia | Una API tiene un costo de tiempo por solicitud; el servidor de desarrollo no está pensado para alta concurrencia |
| Túnel con ngrok (opcional) | Cómo una API local se vuelve accesible desde fuera de esta máquina — el paso previo a un despliegue en la nube real |
| Flask vs. FastAPI, mismo endpoint | Dos formas válidas de resolver el mismo despliegue, con trade-offs distintos (validación automática, documentación interactiva) |

Con este notebook se cierra la experiencia práctica de despliegue: el modelo ya no "vive" solo dentro de un notebook — es un servicio que cualquier aplicación (o, con ngrok, cualquier persona con la URL) puede consultar.
